In [ ]:
import os
import json
import warnings
import numpy as np
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import col, when, mean, row_number
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

# 1. KHỞI TẠO SPARK & DỌN DẸP LOG
warnings.filterwarnings("ignore", category=FutureWarning)
spark = SparkSession.builder.appName("DepressionKNN").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# ============================================================
# 2. ĐỌC & TIỀN XỬ LÝ DỮ LIỆU
# ============================================================
CSV_FILE = "/kaggle/input/datasets/akina484/atn-data-1/Student Depression Dataset.csv"
df = spark.read.csv(CSV_FILE, sep=',', header=True, inferSchema=True)
if len(df.columns) <= 1:
    df = spark.read.csv(CSV_FILE, sep=';', header=True, inferSchema=True)

# Xóa khoảng trắng ở tên cột
for col_name in df.columns:
    df = df.withColumnRenamed(col_name, col_name.strip())

df = df.withColumn("Gender", when(col("Gender") == "Male", 1).otherwise(0))
df = df.withColumn("Family History of Mental Illness",
                   when(col("Family History of Mental Illness") == "Yes", 1).otherwise(0))
df = df.withColumn("Sleep Duration",
        when(col("Sleep Duration") == "Less than 5 hours", 0)
       .when(col("Sleep Duration") == "5-6 hours", 1)
       .when(col("Sleep Duration") == "7-8 hours", 2)
       .otherwise(3))

INPUT_COLS = ["Age", "CGPA", "Academic Pressure", "Study Satisfaction",
              "Financial Stress", "Sleep Duration", "Gender",
              "Family History of Mental Illness"]
existing_cols = [c for c in INPUT_COLS if c in df.columns]

# Ép kiểu dữ liệu và điền giá trị khuyết
for column in existing_cols:
    df = df.withColumn(column, col(column).cast("double"))
    mean_val = df.select(mean(col(column))).collect()[0][0]
    if mean_val is not None:
        df = df.fillna(mean_val, subset=[column])

df = df.withColumn("Depression", col("Depression").cast("integer"))
df.cache()

# ============================================================
# 3. CHUẨN HÓA DỮ LIỆU QUA SPARK PIPELINE (StandardScaler)
# ============================================================
assembler = VectorAssembler(inputCols=existing_cols, outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=True)
prep_pipeline = Pipeline(stages=[assembler, scaler])

prep_model = prep_pipeline.fit(df)
df_scaled = prep_model.transform(df)

# Chuyển features vector thành numpy array
def vector_to_array(v):
    return v.toArray().tolist()

from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, DoubleType

vector_udf = udf(vector_to_array, ArrayType(DoubleType()))
df_scaled = df_scaled.withColumn("features_arr", vector_udf(col("features")))

# ============================================================
# 4. K-FOLD CROSS VALIDATION (K=5) VỚI KNN (sklearn)
# ============================================================
K = 5
KNN_K = 5  # Số láng giềng

print(f"\n🚀 Bắt đầu K-Fold Cross Validation (K={K}, KNN k={KNN_K})...")

windowSpec = Window.orderBy("Age")
df_scaled = df_scaled.withColumn("row_num", row_number().over(windowSpec))
df_scaled = df_scaled.withColumn("fold", (col("row_num") % K).cast("integer"))

fold_accuracies, fold_precisions, fold_recalls, fold_f1s, fold_aucs = [], [], [], [], []

line = "+" + "-"*8 + "+" + "-"*10 + "+" + "-"*11 + "+" + "-"*12 + "+" + "-"*10 + "+" + "-"*10 + "+" + "-"*10 + "+"
header = "| {:<6} | {:<8} | {:<9} | {:<10} | {:<8} | {:<8} | {:<8} |".format(
    "Fold", "Samples", "Accuracy", "Precision", "Recall", "F1-Score", "AUC")

print(line)
print(header)
print(line)

plt.figure(figsize=(8, 6)) # Khởi tạo hình vẽ ROC

for i in range(K):
    train_df = df_scaled.filter(col("fold") != i)
    val_df   = df_scaled.filter(col("fold") == i)

    # Thu thập dữ liệu về driver (KNN chạy trên sklearn)
    train_data = train_df.select("features_arr", "Depression").collect()
    val_data   = val_df.select("features_arr", "Depression").collect()

    X_train = np.array([row["features_arr"] for row in train_data])
    y_train = np.array([row["Depression"]    for row in train_data])
    X_val   = np.array([row["features_arr"]  for row in val_data])
    y_val   = np.array([row["Depression"]    for row in val_data])

    # Huấn luyện KNN
    knn = KNeighborsClassifier(n_neighbors=KNN_K, metric='euclidean', n_jobs=-1)
    knn.fit(X_train, y_train)
    
    # Dự đoán
    y_pred = knn.predict(X_val)
    y_prob = knn.predict_proba(X_val)[:, 1] # Lấy xác suất của lớp 1 (Trầm cảm)

    # Tính toán các chỉ số
    acc = accuracy_score(y_val, y_pred)
    pre = precision_score(y_val, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_val, y_pred, average='weighted', zero_division=0)
    f1  = f1_score(y_val, y_pred, average='weighted', zero_division=0)
    auc_val = roc_auc_score(y_val, y_prob)

    fold_accuracies.append(acc)
    fold_precisions.append(pre)
    fold_recalls.append(rec)
    fold_f1s.append(f1)
    fold_aucs.append(auc_val)

    print("| Fold {:<2} | {:<8} | {:<9.4f} | {:<10.4f} | {:<8.4f} | {:<8.4f} | {:<8.4f} |".format(
        i+1, len(val_data), acc, pre, rec, f1, auc_val))
    
    # Vẽ đường ROC cho Fold hiện tại
    fpr, tpr, _ = roc_curve(y_val, y_prob)
    plt.plot(fpr, tpr, label=f"Fold {i+1} (AUC = {auc_val:.2f})")

stats = {
    "Accuracy":  (np.mean(fold_accuracies), np.std(fold_accuracies)),
    "Precision": (np.mean(fold_precisions), np.std(fold_precisions)),
    "Recall":    (np.mean(fold_recalls),    np.std(fold_recalls)),
    "F1-Score":  (np.mean(fold_f1s),        np.std(fold_f1s)),
    "AUC":       (np.mean(fold_aucs),       np.std(fold_aucs))
}

print(line)
print("| {:<6} | {:<8} | {:<9.4f} | {:<10.4f} | {:<8.4f} | {:<8.4f} | {:<8.4f} |".format(
    "AVG", "-", stats['Accuracy'][0], stats['Precision'][0], stats['Recall'][0], stats['F1-Score'][0], stats['AUC'][0]))
print("| {:<6} | {:<8} | {:<9.4f} | {:<10.4f} | {:<8.4f} | {:<8.4f} | {:<8.4f} |".format(
    "STD", "-", stats['Accuracy'][1], stats['Precision'][1], stats['Recall'][1], stats['F1-Score'][1], stats['AUC'][1]))
print(line)

# ============================================================
# 5. HOÀN THIỆN BIỂU ĐỒ ROC
# ============================================================
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title(f'ROC Curves for Folds (KNN K={KNN_K})')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()
plt.savefig("roc_folds_knn.png", dpi=300, bbox_inches='tight')
print("📊 Đã lưu biểu đồ ROC: roc_folds_knn.png")

# ============================================================
# 6. XUẤT PARAMS (lưu scaler params + KNN config)
# ============================================================
print("\n🧠 Đang lưu model_params.json...")

scaler_model_stage = prep_model.stages[1]
data_json = {
    "model": "KNN",
    "knn_k": KNN_K,
    "metric": "euclidean",
    "features": existing_cols,
    "means": scaler_model_stage.mean.toArray().tolist(),
    "stds":  scaler_model_stage.std.toArray().tolist(),
}
with open("model_params.json", "w") as f:
    json.dump(data_json, f, indent=2)

print("✅ Hoàn tất!")
spark.stop()